In [ ]:
import sys
from pathlib import Path

def _find_scripts_dir():
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        for cand in (base / "scripts", base,
                     *sorted(base.glob("*/scripts")), *sorted(base.glob("*/*/scripts"))):
            if (cand / "project_paths.py").is_file():
                return cand
    raise FileNotFoundError(
        "scripts/project_paths.py not found. Open this notebook from inside the "
        "cloned repository, or point the kernel's working directory at it."
    )

sys.path.insert(0, str(_find_scripts_dir()))
from project_config import *
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr
model_df = load_model_df()
y = model_df["phq9"]
assert len(ACOUSTIC) == 72

In [2]:
# Audio robustness experiment 1: drop 10 random recordings/subject, re-run CV.
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

feat = pd.read_csv(EGEMAPS_CSV, dtype={"subject_id": str}).dropna(subset=["phq9"])

N_DROP = 10
rng = np.random.RandomState(SEED)
kept = pd.concat([g.sample(n=len(g) - N_DROP, random_state=rng) if len(g) > N_DROP else g
                  for _, g in feat.groupby("subject_id")])
print("kept recordings/subject:", kept.groupby("subject_id").size().min(),
      "to", kept.groupby("subject_id").size().max())

X_full,    y_full    = subject_means(feat)      # aligned, ACOUSTIC (module)
X_reduced, y_reduced = subject_means(kept)

models = {"RF": RandomForestRegressor(random_state=SEED),
          "GBM": GradientBoostingRegressor(random_state=SEED)}
print(f"\n{'version':10s}{'model':5s}{'R2':>15}{'MAE':>13}{'RMSE':>13}{'r':>13}")
for vname, (X, yy) in {"Full": (X_full, y_full), "Reduced": (X_reduced, y_reduced)}.items():
    for mname, m in models.items():
        r = repeated_cv(m, X, yy)
        fmt = lambda k: f"{r[k][0]:.3f}±{r[k][1]:.3f}"
        print(f"{vname:10s}{mname:5s}{fmt('R2'):>15}{fmt('MAE'):>13}{fmt('RMSE'):>13}{fmt('r'):>13}")


kept recordings/subject: 14 to 19

version   model             R2          MAE         RMSE            r
Full      RF       0.228±0.032  6.282±0.150  7.380±0.152  0.478±0.032
Full      GBM      0.200±0.067  5.978±0.265  7.505±0.313  0.495±0.048
Reduced   RF       0.145±0.039  6.718±0.172  7.766±0.179  0.389±0.043
Reduced   GBM     -0.023±0.092  7.125±0.386  8.488±0.380  0.286±0.076


In [3]:
# Robustness experiment 2
# Adding white Gaussian noise to every resampled WAV at a target SNR -> parallel tree
import numpy as np, soundfile as sf
from pathlib import Path

clean_root = require(RESAMPLED_DIR, "Run the resampling cell in MODMA_audio_processing.ipynb first.")
noisy_root = NOISY_DIR
SNR_DB = 20                        # 20 dB = mild; 10 dB for a harsher test
rng = np.random.RandomState(SEED)  # reproducible noise

wavs = sorted(clean_root.glob("*/*_resampled.wav"))
for wav in wavs:
    y, sr = sf.read(wav)
    p_signal = np.mean(y ** 2)
    if p_signal == 0:              # skip silent files
        continue
    p_noise = p_signal / (10 ** (SNR_DB / 10))
    y_noisy = y + rng.normal(0.0, np.sqrt(p_noise), size=y.shape)
    out = noisy_root / wav.parent.name / wav.name
    out.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out, y_noisy, sr)
print(f"Wrote {len(wavs)} noisy files at SNR={SNR_DB} dB to {noisy_root}")

Wrote 1475 noisy files at SNR=20 dB to C:\Users\zeine\thesis\depression-severity\audio_lanzhou_2015\audio_lanzhou_2015_noisy


In [4]:
# Parse the NOISY eGeMAPS ARFFs -> egemaps_features_noisy.csv
import csv as _csv
from pathlib import Path
import pandas as pd

features_root   = NOISY_DIR             # <-- noisy tree
OUTPUT_FEATURES = EGEMAPS_NOISY_CSV     # <-- new output file

def parse_arff(path: Path) -> dict:
    attributes, data_line, in_data = [], None, False
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if not in_data:
                low = s.lower()
                if low.startswith("@attribute"):
                    attributes.append(s.split()[1])
                elif low.startswith("@data"):
                    in_data = True
            elif data_line is None:
                data_line = s
    if data_line is None:
        raise ValueError(f"No @data row in {path}")
    values = next(_csv.reader([data_line], quotechar="'"))
    row = {}
    for attr, val in zip(attributes, values):
        if attr in ("name", "class"):
            continue
        row[attr] = pd.NA if val == "?" else float(val)
    return row

# Subject-level clinical data (type, PHQ-9) from the inventory.
inv = pd.read_csv(WAV_INVENTORY, dtype={"subject_id": str})
clinical = (
    inv[["subject_id", "type", "phq9"]]
    .drop_duplicates("subject_id")
    .set_index("subject_id")
)

rows = []
for arff in sorted(features_root.glob("*/egemaps/*_egemaps.*")):
    subject_id = arff.parents[1].name
    recording = arff.stem.replace("_resampled_egemaps", "")
    feats = parse_arff(arff)
    meta = clinical.loc[subject_id] if subject_id in clinical.index else None
    rows.append({
        "subject_id": subject_id,
        "recording": recording,
        "type": meta["type"] if meta is not None else pd.NA,
        "phq9": meta["phq9"] if meta is not None else pd.NA,
        **feats,
    })

features = pd.DataFrame(rows).sort_values(["subject_id", "recording"]).reset_index(drop=True)
features.to_csv(OUTPUT_FEATURES, index=False)
print(f"Parsed {len(features)} noisy ARFF files -> {OUTPUT_FEATURES}")


Parsed 1475 noisy ARFF files -> C:\Users\zeine\thesis\depression-severity\audio_lanzhou_2015\egemaps_features_noisy.csv


In [5]:
# Clean vs Noisy comparison on the 72-feature (collapsed) acoustic set
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Module subject_table: defaults to ACOUSTIC (72) and aligns to CANONICAL_ORDER
versions = {
    "Clean (72)": subject_table("egemaps_features.csv"),
    "Noisy (72)": subject_table("egemaps_features_noisy.csv"),
}
models = {"RF": RandomForestRegressor(random_state=SEED),
          "GBM": GradientBoostingRegressor(random_state=SEED)}

print(f"{'version':12s}{'model':5s}{'R2':>15}{'MAE':>13}{'RMSE':>13}{'r':>13}")
for vname, (X, y) in versions.items():
    for mname, m in models.items():
        r = repeated_cv(m, X, y)                 # module repeated_cv -> {metric:(mean,std)}
        fmt = lambda k: f"{r[k][0]:.3f}±{r[k][1]:.3f}"
        print(f"{vname:12s}{mname:5s}{fmt('R2'):>15}{fmt('MAE'):>13}{fmt('RMSE'):>13}{fmt('r'):>13}")



version     model             R2          MAE         RMSE            r
Clean (72)  RF       0.228±0.032  6.282±0.150  7.380±0.152  0.478±0.032
Clean (72)  GBM      0.200±0.067  5.978±0.265  7.505±0.313  0.495±0.048
Noisy (72)  RF       0.173±0.044  6.512±0.192  7.638±0.199  0.424±0.042
Noisy (72)  GBM      0.013±0.099  6.858±0.375  8.336±0.414  0.345±0.075


In [6]:
# Psych robustness experiment 1: drop GAD-7, compare to full psych scales
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr

y = model_df["phq9"]
psych_full = ["ctq_sf", "LES", "SSRS", "gad7", "PSQI"]
psych_drop = [c for c in psych_full if c != "gad7"]        # perturbation: GAD-7 removed

def evaluate(model, X, y, seeds=range(20)):
    rows = []
    for s in seeds:
        kf = KFold(n_splits=10, shuffle=True, random_state=s)
        p = cross_val_predict(model, X, y, cv=kf, n_jobs=-1)
        rows.append({"MAE": mean_absolute_error(y, p), "RMSE": np.sqrt(mean_squared_error(y, p)),
                     "R2": r2_score(y, p), "r": pearsonr(y, p)[0]})
    return pd.DataFrame(rows).mean()          # mean across the 20 repeats

models = {"RF": RandomForestRegressor(random_state=SEED),
          "GBM": GradientBoostingRegressor(random_state=SEED)}

rows = []
for mname, m in models.items():
    full = evaluate(m, model_df[psych_full], y)
    drop = evaluate(m, model_df[psych_drop], y)
    for metric in ["MAE", "RMSE", "R2", "r"]:
        rows.append({
            "model": mname, "metric": metric,
            "original": round(full[metric], 3),
            "perturbed": round(drop[metric], 3),
            "delta": round(drop[metric] - full[metric], 3),
        })

comp = pd.DataFrame(rows)
# For MAE/RMSE, higher = worse; for R2/r, lower = worse. Express as a signed "% worse".
def pct_worse(row):
    o, p = row["original"], row["perturbed"]
    if row["metric"] in ("MAE", "RMSE"):
        return round(100 * (p - o) / o, 1)          # positive = degraded
    return round(100 * (o - p) / abs(o), 1)         # positive = degraded
comp["pct_worse"] = comp.apply(pct_worse, axis=1)

print(comp.to_string(index=False))
print("\nMost-degraded metric per model:")
for mname in models:
    sub = comp[comp["model"] == mname]
    worst = sub.loc[sub["pct_worse"].idxmax()]
    print(f"  {mname}: {worst['metric']} degraded {worst['pct_worse']}% "
          f"({worst['original']} -> {worst['perturbed']})")


model metric  original  perturbed  delta  pct_worse
   RF    MAE     2.561      3.196  0.635       24.8
   RF   RMSE     3.705      4.420  0.715       19.3
   RF     R2     0.805      0.722 -0.083       10.3
   RF      r     0.899      0.851 -0.047        5.3
  GBM    MAE     2.706      3.175  0.469       17.3
  GBM   RMSE     3.750      4.315  0.565       15.1
  GBM     R2     0.799      0.735 -0.064        8.0
  GBM      r     0.897      0.862 -0.035        3.9

Most-degraded metric per model:
  RF: MAE degraded 24.8% (2.561 -> 3.196)
  GBM: MAE degraded 17.3% (2.706 -> 3.175)


In [7]:
# Psych robustness experiment 2: random partial missingness (MCAR) + imputation
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr

psych = ["ctq_sf", "LES", "SSRS", "gad7", "PSQI"]
X_psych = model_df[psych].astype(float)
y = model_df["phq9"]

MISSING_FRACS = [0.0, 0.1, 0.2, 0.3, 0.5]      # 0.0 = original baseline
rng = np.random.RandomState(SEED)              # reproducible missingness

def add_missing(X, frac):
    """Set a random `frac` of cells to NaN (missing completely at random)."""
    Xm = X.copy()
    mask = rng.rand(*Xm.shape) < frac
    return Xm.mask(mask)                        # True positions -> NaN

def evaluate(pipe, X, y, seeds=range(20)):
    rows = []
    for s in seeds:
        kf = KFold(n_splits=10, shuffle=True, random_state=s)
        p = cross_val_predict(pipe, X, y, cv=kf, n_jobs=-1)
        rows.append({"MAE": mean_absolute_error(y, p), "RMSE": np.sqrt(mean_squared_error(y, p)),
                     "R2": r2_score(y, p), "r": pearsonr(y, p)[0]})
    return pd.DataFrame(rows).mean()

models = {"RF": RandomForestRegressor(random_state=SEED),
          "GBM": GradientBoostingRegressor(random_state=SEED)}

rows = []
for mname, m in models.items():
    for frac in MISSING_FRACS:
        Xm = add_missing(X_psych, frac)
        # median imputation fit inside each CV fold (no leakage)
        pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("model", m)])
        res = evaluate(pipe, Xm, y)
        rows.append({"model": mname, "missing_%": int(frac * 100),
                     "MAE": round(res["MAE"], 3), "RMSE": round(res["RMSE"], 3),
                     "R2": round(res["R2"], 3), "r": round(res["r"], 3)})

comp = pd.DataFrame(rows)
print(comp.to_string(index=False))

# Degradation vs the 0% baseline, and which metric degrades most (per model)
print("\nDegradation vs 0% missing (positive = worse):")
for mname in models:
    base = comp[(comp.model == mname) & (comp["missing_%"] == 0)].iloc[0]
    worst = comp[(comp.model == mname) & (comp["missing_%"] == 50)].iloc[0]  # at heaviest missingness
    deg = {
        "MAE":  round(100 * (worst.MAE  - base.MAE)  / base.MAE,  1),
        "RMSE": round(100 * (worst.RMSE - base.RMSE) / base.RMSE, 1),
        "R2":   round(100 * (base.R2 - worst.R2) / abs(base.R2), 1),
        "r":    round(100 * (base.r  - worst.r)  / abs(base.r),  1),
    }
    worst_metric = max(deg, key=deg.get)
    print(f"  {mname} @50% missing: {deg}  -> most degraded: {worst_metric} ({deg[worst_metric]}%)")


model  missing_%   MAE  RMSE    R2     r
   RF          0 2.561 3.705 0.805 0.899
   RF         10 2.689 3.999 0.773 0.883
   RF         20 3.586 5.110 0.629 0.798
   RF         30 3.479 4.818 0.670 0.823
   RF         50 3.897 5.164 0.621 0.791
  GBM          0 2.706 3.750 0.799 0.897
  GBM         10 3.499 5.039 0.639 0.807
  GBM         20 2.938 4.276 0.739 0.863
  GBM         30 3.296 4.559 0.702 0.843
  GBM         50 5.048 6.793 0.344 0.624

Degradation vs 0% missing (positive = worse):
  RF @50% missing: {'MAE': np.float64(52.2), 'RMSE': np.float64(39.4), 'R2': np.float64(22.9), 'r': np.float64(12.0)}  -> most degraded: MAE (52.2%)
  GBM @50% missing: {'MAE': np.float64(86.5), 'RMSE': np.float64(81.1), 'R2': np.float64(56.9), 'r': np.float64(30.4)}  -> most degraded: MAE (86.5%)


In [8]:
# Experiments 1 & 2 for Demographics and Clinical (full)
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr

y = model_df["phq9"]
feature_sets = {
    "Demographics": ["age", "gender", "education_years"],
    "Clinical":     ["age", "gender", "education_years", "ctq_sf", "LES", "SSRS", "gad7", "PSQI"],
}
models = {"RF": RandomForestRegressor(random_state=SEED),
          "GBM": GradientBoostingRegressor(random_state=SEED)}

def evaluate(est, X, y, seeds=range(20)):
    rows = []
    for s in seeds:
        kf = KFold(n_splits=10, shuffle=True, random_state=s)
        p = cross_val_predict(est, X, y, cv=kf, n_jobs=-1)
        rows.append({"MAE": mean_absolute_error(y, p), "RMSE": np.sqrt(mean_squared_error(y, p)),
                     "R2": r2_score(y, p), "r": pearsonr(y, p)[0]})
    return pd.DataFrame(rows).mean()

def pct_worse(metric, base, pert):
    if metric in ("MAE", "RMSE"):
        return round(100 * (pert - base) / base, 1)          # error up = worse
    return round(100 * (base - pert) / abs(base), 1)         # score down = worse

def top_feature(cols):
    rf = RandomForestRegressor(random_state=SEED).fit(model_df[cols], y)
    return pd.Series(rf.feature_importances_, index=cols).idxmax()

# ---------- Experiment 1: drop the most important feature ----------
print("=== EXPERIMENT 1: drop most-important feature ===")
for sname, cols in feature_sets.items():
    drop_feat = top_feature(cols)
    reduced = [c for c in cols if c != drop_feat]
    for mname, m in models.items():
        base = evaluate(m, model_df[cols], y)
        pert = evaluate(m, model_df[reduced], y)
        deg = {k: pct_worse(k, base[k], pert[k]) for k in ["MAE", "RMSE", "R2", "r"]}
        worst = max(deg, key=deg.get)
        print(f"{sname:12s} {mname:3s} drop '{drop_feat}':  "
              f"R2 {base['R2']:.3f}->{pert['R2']:.3f}  MAE {base['MAE']:.3f}->{pert['MAE']:.3f}  "
              f"| most degraded: {worst} ({deg[worst]}%)")

# ---------- Experiment 2: random partial missingness (MCAR) + imputation ----------
print("\n=== EXPERIMENT 2: random missingness ===")
MISSING_FRACS = [0.0, 0.1, 0.2, 0.3, 0.5]
rng = np.random.RandomState(SEED)

def add_missing(X, frac):
    return X.mask(rng.rand(*X.shape) < frac)

for sname, cols in feature_sets.items():
    X = model_df[cols].astype(float)
    for mname, m in models.items():
        base = None
        for frac in MISSING_FRACS:
            Xm = add_missing(X, frac)
            pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("model", m)])
            res = evaluate(pipe, Xm, y)
            if frac == 0.0:
                base = res
            tag = "(baseline)" if frac == 0 else ""
            print(f"{sname:12s} {mname:3s} {int(frac*100):3d}% missing:  "
                  f"R2 {res['R2']:.3f}  MAE {res['MAE']:.3f}  RMSE {res['RMSE']:.3f}  r {res['r']:.3f} {tag}")
        # degradation at 50% vs baseline
        Xm = add_missing(X, 0.5)
        print()


=== EXPERIMENT 1: drop most-important feature ===
Demographics RF  drop 'age':  R2 0.188->0.183  MAE 5.964->5.899  | most degraded: r (4.2%)
Demographics GBM drop 'age':  R2 0.045->0.122  MAE 6.294->6.117  | most degraded: r (3.0%)
Clinical     RF  drop 'gad7':  R2 0.784->0.700  MAE 2.663->3.304  | most degraded: MAE (24.1%)
Clinical     GBM drop 'gad7':  R2 0.765->0.681  MAE 2.766->3.406  | most degraded: MAE (23.1%)

=== EXPERIMENT 2: random missingness ===
Demographics RF    0% missing:  R2 0.188  MAE 5.964  RMSE 7.564  r 0.495 (baseline)
Demographics RF   10% missing:  R2 0.131  MAE 6.221  RMSE 7.827  r 0.444 
Demographics RF   20% missing:  R2 -0.144  MAE 7.574  RMSE 8.980  r 0.159 
Demographics RF   30% missing:  R2 0.149  MAE 5.718  RMSE 7.746  r 0.463 
Demographics RF   50% missing:  R2 -0.020  MAE 6.849  RMSE 8.477  r 0.265 

Demographics GBM   0% missing:  R2 0.045  MAE 6.294  RMSE 8.203  r 0.452 (baseline)
Demographics GBM  10% missing:  R2 -0.237  MAE 7.320  RMSE 9.332  r 0

In [ ]:
# ===== Robustness summary: original vs perturbed, all modalities =====
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Local scorer keeps the median-imputer pipeline (needed for the missingness rows)
def score(X, y, seeds=REPEATED_CV_SEEDS):
    r2, mae = [], []
    for s in seeds:
        kf = KFold(n_splits=10, shuffle=True, random_state=s)
        est = Pipeline([("imp", SimpleImputer(strategy="median")),
                        ("rf", RandomForestRegressor(random_state=SEED))])
        p = cross_val_predict(est, X, y, cv=kf, n_jobs=1)      # n_jobs=1: dual-sklearn safe
        r2.append(r2_score(y, p)); mae.append(mean_absolute_error(y, p))
    return np.mean(r2), np.mean(mae)

rng = np.random.RandomState(SEED)
clin_sets = {"Demographics": DEMO, "Psych": PSYCH, "Clinical": CLINICAL}

records = []
def add(modality, perturbation, r2o, maeo, r2p, maep):
    records.append({
        "modality": modality, "perturbation": perturbation,
        "R2_orig": round(r2o, 3), "R2_pert": round(r2p, 3),
        "R2_deg_%": round(100 * (r2o - r2p) / abs(r2o), 1),
        "MAE_orig": round(maeo, 3), "MAE_pert": round(maep, 3),
        "MAE_deg_%": round(100 * (maep - maeo) / maeo, 1),
    })

# --- Clinical-family perturbations: drop-top-feature & 30% missingness ---
for name, cols in clin_sets.items():
    r2o, maeo = score(model_df[cols], y)
    top = pd.Series(RandomForestRegressor(random_state=SEED)
                    .fit(model_df[cols], y).feature_importances_, index=cols).idxmax()
    r2d, maed = score(model_df[[c for c in cols if c != top]], y)
    add(name, f"drop '{top}'", r2o, maeo, r2d, maed)
    Xm = model_df[cols].astype(float).mask(rng.rand(len(model_df), len(cols)) < 0.30)
    r2m, maem = score(Xm, y)
    add(name, "30% missing", r2o, maeo, r2m, maem)

# --- Acoustic perturbations: drop-10-recordings & noise (aligned via subject_means) ---
feat = pd.read_csv(DATA_DIR / "egemaps_features.csv", dtype={"subject_id": str}).dropna(subset=["phq9"])
Xa, ya = subject_means(feat)                          # aligned, ACOUSTIC
r2o, maeo = score(Xa, ya)

kept = pd.concat([g.sample(n=max(len(g) - 10, 1), random_state=rng) for _, g in feat.groupby("subject_id")])
Xd, yd = subject_means(kept)
r2d, maed = score(Xd, yd)
add("Acoustic", "drop 10 recordings", r2o, maeo, r2d, maed)

if (DATA_DIR / "egemaps_features_noisy.csv").exists():
    Xn, yn = subject_table("egemaps_features_noisy.csv")   # module, aligned
    r2n, maen = score(Xn, yn)
    add("Acoustic", "noise (SNR)", r2o, maeo, r2n, maen)
else:
    print("(noisy features not found yet — skipping the noise row)")

# --- Compact table + stability verdict ---
summary = pd.DataFrame(records)
print(summary.to_string(index=False))
summary.to_csv(DATA_DIR / "robustness_summary.csv", index=False)
stability = summary.groupby("modality")["R2_deg_%"].mean().sort_values().round(1)
print("\nMean R2 degradation by modality (lower = more stable):")
print(stability.to_string())
print(f"\nMost stable: {stability.index[0]}  |  Least stable: {stability.index[-1]}")

